In [9]:
import pandas as pd
from pathlib import Path

DATA = Path("..") /"Data"/ "kenya_retail"

tx = pd.read_csv(DATA / "transactions.csv", parse_dates=["date"])
stores = pd.read_csv(DATA / "stores.csv")
customers = pd.read_csv(DATA / "customers.csv", parse_dates=["signup_date"])
products = pd.read_csv(DATA / "products.csv")

for name, df in [("transactions", tx), ("stores", stores),
                 ("customers", customers), ("products", products)]:
    print(name, df.shape)

transactions (1802309, 16)
stores (60, 6)
customers (2280, 7)
products (62849, 5)


In [10]:
print("duplicate rows:", tx.duplicated().sum())
tx = tx.drop_duplicates()

print("missing payment method:", tx["payment_method"].isna().sum())
tx["payment_method"] = tx["payment_method"].fillna("Unknown")

print("missing customer ID:", tx["customer_id"].isna().mean().round(3))

duplicate rows: 3581
missing payment method: 5407
missing customer ID: 0.079


In [11]:
df = (tx
      .merge(stores.rename(columns={"county": "store_county"}), on="store_id", how="left")
      .merge(products, on="product_id", how="left")
      .merge(customers[["customer_id", "county", "loyalty_tier"]]
             .rename(columns={"county": "customer_county"}),
             on="customer_id", how="left"))
df.head()

,line_id,basket_id,date,hour,store_id,customer_id,product_id,quantity,unit_price_kes,gross_sales_kes,...,tier,format,size_sqm,opened_year,category,product_group,product_subgroup,brand,customer_county,loyalty_tier
0,1,26984851472,2024-01-01,16,KE-ST042,C02166,1004906,1,140.0,140.0,...,Rural,Mini-market,448,2013,PRODUCE,POTATOES,POTATOES RUSSET (BULK&BAG),Private,Bungoma,Silver
1,2,26984851472,2024-01-01,16,KE-ST042,C02166,1033142,1,60.0,60.0,...,Rural,Mini-market,448,2013,PRODUCE,ONIONS,ONIONS SWEET (BULK&BAG),National,Bungoma,Silver
2,3,26984851472,2024-01-01,16,KE-ST042,C02166,1036325,1,90.0,90.0,...,Rural,Mini-market,448,2013,PRODUCE,VEGETABLES - ALL OTHERS,CELERY,Private,Bungoma,Silver
3,4,26984851472,2024-01-01,16,KE-ST042,C02166,1082185,1,85.0,85.0,...,Rural,Mini-market,448,2013,PRODUCE,TROPICAL FRUIT,BANANAS,National,Bungoma,Silver
4,5,26984851472,2024-01-01,16,KE-ST042,C02166,8160430,1,135.0,135.0,...,Rural,Mini-market,448,2013,PRODUCE,ORGANICS FRUIT & VEGETABLES,ORGANIC CARROTS,Private,Bungoma,Silver


In [12]:
df.groupby("store_county")["net_sales_kes"].sum().sort_values(ascending=False).head(10)

store_county
Nairobi        58858289.0
Bungoma        51605051.0
Kakamega       37379286.0
Machakos       34665143.0
Kiambu         28740040.0
Trans Nzoia    20089244.0
Laikipia       18918355.0
Uasin Gishu    17145132.0
Kisumu         16619146.0
Murang'a       16517704.0
Name: net_sales_kes, dtype: float64